In [1]:
import pandas as pd, numpy as np
from pprint import pprint
import csv

from video_game_retention.api import (
    get_steam_game_details,
    get_steam_player_history,
    get_top_games
)
from video_game_retention.config import PROJECT_ROOT

In [ ]:

# Grand Theft Auto V Legacy

game = get_steam_game_details(271590)
history = get_steam_player_history(271590)

points = history["points"]

print("Game:", game["name"])
print("Release date:", game["releaseDate"])
print("History resolution:", history["resolution"])
print("Observations:", len(points))

if points:
    print("Earliest observation:", min(p["at"] for p in points))
    print("Latest observation:", max(p["at"] for p in points))

In [ ]:
# variable type
for key, value in game.items():
    print(f"{key}: {type(value).__name__}")

In [ ]:
pprint(game)

In [ ]:
app_ids = [
    730,      # Counter-Strike 2
    1086940,  # Baldur's Gate 3
    1091500,  # Cyberpunk 2077
    271590,
]

results = []

for app_id in app_ids:
    game = get_steam_game_details(app_id)
    history = get_steam_player_history(app_id, "all")

    points = history.get("points", [])

    results.append({
        "app_id": app_id,
        "name": game["name"],
        "release_date": game["releaseDate"],
        "first_observation": min(
            (p["at"] for p in points),
            default=None,
        ),
        "last_observation": max(
            (p["at"] for p in points),
            default=None,
        ),
        "observations": len(points),
        "resolution": history.get("resolution"),
    })

df = pd.DataFrame(results)
display(df)

Earliest observation: 2023-08-01T00:00:00Z

In [2]:
# get the top 50 games from SteamIDX -- can me change later
top_games = get_top_games(100)

print(type(top_games))
print(len(top_games))

<class 'list'>
100


In [3]:
from pprint import pprint

pprint(top_games[:10])

[{'allTimePeak': 1450705,
  'appId': 730,
  'currentPlayers': 1030291,
  'name': 'Counter-Strike 2',
  'peak24h': 1221105,
  'rank': 1,
  'type': 'game',
  'url': 'https://steamidx.com/app/730'},
 {'allTimePeak': 1097854,
  'appId': 570,
  'currentPlayers': 685686,
  'name': 'Dota 2',
  'peak24h': 728011,
  'rank': 2,
  'type': 'game',
  'url': 'https://steamidx.com/app/570'},
 {'allTimePeak': 549324,
  'appId': 4262460,
  'currentPlayers': 509951,
  'name': 'ROB RUN',
  'peak24h': 549324,
  'rank': 3,
  'type': 'game',
  'url': 'https://steamidx.com/app/4262460'},
 {'allTimePeak': 816367,
  'appId': 578080,
  'currentPlayers': 223717,
  'name': 'PUBG: BATTLEGROUNDS',
  'peak24h': 755830,
  'rank': 4,
  'type': 'game',
  'url': 'https://steamidx.com/app/578080'},
 {'allTimePeak': 397905,
  'appId': 3393110,
  'currentPlayers': 144683,
  'name': 'AION 2',
  'peak24h': 149613,
  'rank': 5,
  'type': 'game',
  'url': 'https://steamidx.com/app/3393110'},
 {'allTimePeak': 207640,
  'appId':

In [4]:
# load fronkon 
csv_path = f"{PROJECT_ROOT}/data/raw/games.csv"

with open(csv_path, encoding="utf-8", newline="") as f:
    header = next(csv.reader(f))

corrected_header = (
    header[:7]
    + ["Discount", "DLC count"]
    + header[8:]
)

fronkon_df = pd.read_csv(
    csv_path,
    header=0,
    names=corrected_header,
    index_col=False
)

fronkon_ids = (
    pd.to_numeric(fronkon_df['AppID'], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna(how='all').astype(int).drop_duplicates().tolist()
)

In [5]:
top_df = pd.DataFrame(top_games)

top_df["in_fronkon"] = top_df["appId"].isin(fronkon_ids)

print("Already in FronkonGames:", top_df["in_fronkon"].sum())
print("Missing from FronkonGames:", (~top_df["in_fronkon"]).sum())

Already in FronkonGames: 84
Missing from FronkonGames: 16


In [6]:
# GTA V legacy in?
print("In FronkonGames:", 271590 in fronkon_ids)

In FronkonGames: True


In [7]:
print("In SteamIDX Top 50:", 271590 in top_df["appId"].astype(int).tolist())

In SteamIDX Top 50: True


In [8]:
top_df[~top_df["in_fronkon"]][
    ["appId", "name"]
]

,appId,name
2,4262460,ROB RUN
4,3393110,AION 2
6,4115450,Phantom Blade Zero
9,2868840,Slay the Spire 2
15,1867240,WARDOGS
16,4530370,Football Manager 27
21,3678970,TBH: Task Bar Hero
27,4080220,EA SPORTS FC™ 27
29,4348910,Probably Stolen
31,480,Spacewar
